# ⚙️ 高度な実装ガイド - RAG / ファインチューニング / 自律性評価
**理論から実装まで完全網羅**

---

## 📚 目次
1. [Simple RAG (TF-IDF ベース)](#1-simple-rag)
2. [Vector RAG (Sentence Transformers)](#2-vector-rag)
3. [ファインチューニング](#3-ファインチューニング)
4. [自律性スコアラー](#4-自律性スコアラー)
5. [フィードバックループ](#5-フィードバックループ)

## 1. Simple RAG (TF-IDF ベース)

キーワードマッチングで関連文書を検索し、LLM が回答を生成

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class SimpleRAG:
    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.knowledge_base = [
            "Machine learning is a subset of artificial intelligence that uses data to learn.",
            "Transformers use self-attention mechanism for processing sequential data.",
            "FAISS is a library for efficient similarity search of dense vectors.",
            "RAG combines retrieval and generation to ground LLM answers in facts.",
            "Fine-tuning adapts a pretrained model to a specific task with labeled data.",
        ]
        self.knowledge_vectors = self.vectorizer.fit_transform(self.knowledge_base)

    def retrieve(self, query, top_k=2):
        qv = self.vectorizer.transform([query])
        sims = cosine_similarity(qv, self.knowledge_vectors)[0]
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.knowledge_base[i], sims[i]) for i in top_idx]

    def answer(self, query):
        docs = self.retrieve(query)
        print(f"\n質問: {query}")
        print("─" * 60)
        print("検索された関連文書:")
        for doc, score in docs:
            print(f"  [{score:.4f}] {doc}")
        context = " ".join([d for d, _ in docs])
        print(f"\n回答ヒント (コンテキスト): {context[:100]}...")
        return docs

rag = SimpleRAG()
for q in ["What is machine learning?", "How does FAISS work?", "Explain fine-tuning"]:
    rag.answer(q)

## 2. Vector RAG (Sentence Transformers)

より高精度なセマンティック検索

In [ ]:
# sentence-transformers がインストール済みの場合のみ実行
try:
    from sentence_transformers import SentenceTransformer
    import torch

    class VectorRAG:
        def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
            self.model = SentenceTransformer(model_name)
            self.knowledge_base = [
                "Python is a high-level programming language.",
                "Machine learning is a subset of artificial intelligence.",
                "Deep learning uses neural networks with multiple layers.",
                "Natural language processing deals with text analysis.",
                "Transformers are neural network architectures based on attention.",
            ]
            self.kb_embeddings = self.model.encode(self.knowledge_base, convert_to_tensor=True)

        def retrieve(self, query, top_k=2):
            qe = self.model.encode(query, convert_to_tensor=True)
            sims = torch.nn.functional.cosine_similarity(qe.unsqueeze(0), self.kb_embeddings)
            top_idx = torch.argsort(sims, descending=True)[:top_k]
            return [(self.knowledge_base[i], sims[i].item()) for i in top_idx]

    vrag = VectorRAG()
    for q in ["What is deep learning?", "Tell me about NLP", "How do transformers work?"]:
        docs = vrag.retrieve(q)
        print(f"\n質問: {q}")
        for doc, score in docs:
            print(f"  [{score:.4f}] {doc}")

except ImportError:
    print("sentence-transformers が未インストールです。")
    print("pip install sentence-transformers  でインストールしてください。")

## 3. ファインチューニング

事前学習済みモデルを特定タスクに適応させる

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.enc = tokenizer(texts, truncation=True, padding="max_length",
                             max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

# 感情分類の模擬ファインチューニング
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

train_texts = ["I love this!", "Amazing product!", "Worst experience", "Terrible service", "Great work!", "Bad quality"]
train_labels = [1, 1, 0, 0, 1, 0]

dataset = TextDataset(train_texts, train_labels, tok)
loader  = DataLoader(dataset, batch_size=2, shuffle=True)
optim   = torch.optim.AdamW(model.parameters(), lr=2e-5)

# 3 エポックだけ実行（デモ用）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(3):
    total_loss = 0
    for batch, labels in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = labels.to(device)
        out  = model(**batch, labels=labels)
        loss = out.loss
        optim.zero_grad(); loss.backward(); optim.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}  loss={total_loss/len(loader):.4f}")

## 4. 自律性スコアラー

9 つの評価軸で AI エージェントの自律度を測定

In [ ]:
class AutonomyScorer:
    CRITERIA = [
        "planning",      # 計画性
        "learning",      # 学習性
        "adaptation",    # 適応性
        "decision",      # 判断性
        "explainability",# 説明性
        "efficiency",    # 効率性
        "reliability",   # 信頼性
        "robustness",    # 頑健性
        "exploration",   # 探索性
    ]

    def __init__(self):
        self.scores = {c: 0.0 for c in self.CRITERIA}

    def score_planning(self, goals: list, steps: list):
        s = (30 if goals else 0) + (70 * min(len(steps)/5, 1))
        self.scores["planning"] = s; return s

    def score_learning(self, history: list):
        if len(history) < 2:
            self.scores["learning"] = 0; return 0
        imp = (history[-1] - history[0]) / (history[0] + 1e-9) * 100
        s = min(max(imp * 2, 0), 100)
        self.scores["learning"] = s; return s

    def score_explainability(self, explanations: list):
        if not explanations: self.scores["explainability"] = 0; return 0
        s = sum(100 if e and len(e) > 20 else 0 for e in explanations) / len(explanations)
        self.scores["explainability"] = s; return s

    def overall(self):
        return sum(self.scores.values()) / len(self.scores)

    def report(self):
        print(f"{'評価軸':<16} スコア  グラフ")
        print("─" * 50)
        for c, v in self.scores.items():
            bar = "▓" * int(v / 5)
            print(f"{c:<16} {v:5.1f}  {bar}")
        print("─" * 50)
        print(f"{'総合スコア':<16} {self.overall():5.1f}")

scorer = AutonomyScorer()
scorer.score_planning(goals=["答えを出す", "品質維持"], steps=["クエリ解析","検索","回答生成","検証","出力"])
scorer.score_learning(history=[0.50, 0.62, 0.71, 0.78, 0.83])
scorer.score_explainability(["この回答は○○の資料を基に生成しました", "検索スコアが高い上位3件を参照しました"])
# 残りを仮置き
for k in ["adaptation","decision","efficiency","reliability","robustness","exploration"]:
    scorer.scores[k] = 65.0

scorer.report()

## 5. フィードバックループ

エージェントが評価結果を使って改善するサイクル

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# フィードバックループのシミュレーション
np.random.seed(0)
rounds = 20

scores = [50.0]
for i in range(1, rounds):
    improvement = np.random.uniform(0.5, 3.5) * (100 - scores[-1]) / 100
    noise = np.random.normal(0, 0.5)
    scores.append(min(scores[-1] + improvement + noise, 98))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(scores, "o-", color="#4dabf7", lw=2)
ax1.fill_between(range(len(scores)), scores, alpha=0.15)
ax1.set_xlabel("フィードバックラウンド")
ax1.set_ylabel("自律性スコア")
ax1.set_title("フィードバックによる自律性スコアの改善")
ax1.axhline(80, color="red", ls="--", label="目標ライン 80")
ax1.legend()

deltas = np.diff(scores)
ax2.bar(range(1, len(deltas)+1), deltas,
        color=["#51cf66" if d > 0 else "#fa5252" for d in deltas])
ax2.set_xlabel("ラウンド"); ax2.set_ylabel("スコア変化")
ax2.set_title("各ラウンドの改善量")

plt.tight_layout(); plt.show()

print(f"初期スコア: {scores[0]:.1f}")
print(f"最終スコア: {scores[-1]:.1f}")
print(f"改善幅   : +{scores[-1]-scores[0]:.1f} ({(scores[-1]-scores[0])/scores[0]*100:.1f}%)")

## ✅ 理解度チェック

- [ ] TF-IDF と Vector RAG の違いが説明できる  
- [ ] ファインチューニングの主要パラメータ (lr, epochs) の役割が分かった  
- [ ] AutonomyScorer の 9 評価軸が何を測っているか説明できる  
- [ ] フィードバックループがスコアを向上させることを確認した  

---
✅ 全ガイド完了 → 実際の [autonomous_rag_agent.py](../autonomous_rag_agent.py) を読んでみよう！